# Pipeline Enrich Bằng Geoapify Places API Cho Bộ 100 Listing

Notebook này enrich bộ dữ liệu `100` nhà riêng ở Gò Vấp và Tân Bình bằng `Geoapify Places API`.

Mục tiêu:
- đọc file `data/go_vap_tan_binh_100.json`
- gọi `Geoapify Places API` theo `latitude/longitude` của từng BĐS
- sinh các feature khoảng cách và số lượng POI xung quanh
- lưu output vào thư mục `data/geoapify/`

Ghi chú:
- Cần API key, truyền qua biến môi trường `GEOAPIFY_API_KEY`.
- Theo tài liệu chính thức, Places API dùng HTTP GET qua endpoint `https://api.geoapify.com/v2/places`.
- Free plan hiện có `3000 credits/day`, và giới hạn gói Free là `up to 5 requests/second`.
- Mỗi request Places API tốn credit theo số lượng kết quả trả về; cứ mỗi `20` places là `1 credit`.
- Notebook này dùng batch + checkpoint để tiện resume và kiểm soát quota.


## Chạy Nhanh

Notebook này dành cho pipeline `Geoapify Places API` hiện tại của nhóm.

Input chính:
- `data/go_vap_tan_binh_100.json`

Output chính:
- `data/geoapify/go_vap_tan_binh_100_enriched_geoapify_api.json`
- `data/geoapify/go_vap_tan_binh_100_enriched_geoapify_api_checkpoint.json`
- `data/geoapify/go_vap_tan_binh_100_geoapify_errors.json`
- `data/geoapify/cache/`

Điều kiện chạy:
- Cần có `GEOAPIFY_API_KEY` trong shell hoặc `.env`
- Nếu có `python-dotenv` thì notebook sẽ tự load `.env`
- Nếu không có `python-dotenv`, chỉ cần `export GEOAPIFY_API_KEY=...` trước khi chạy

Tài liệu liên quan:
- `data/geoapify/geoapify_enriched_schema_readme.json`
- `data/geoapify/geoapify_api_curl_examples.md`
- `.env.example`
- `docs/provider_comparison_overpass_vs_geoapify.md`


## Bước 0. Khởi Tạo Đường Dẫn Và API Key

Mục đích:
- Khai báo đường dẫn input/output cho pipeline Geoapify.
- Đọc `GEOAPIFY_API_KEY` từ biến môi trường hoặc file `.env`.
- Tạo thư mục `data/geoapify/` và cache bên trong nếu chưa có.

Input:
- File dữ liệu gốc trong thư mục `data/`.
- Biến môi trường `GEOAPIFY_API_KEY` hoặc file `.env` ở thư mục gốc project.

Output:
- Các biến `INPUT_PATH`, `OUTPUT_PATH`, `CHECKPOINT_PATH`, `ERROR_LOG_PATH`, `CACHE_DIR`, `GEOAPIFY_API_KEY`.

Ghi chú thêm:
- Nếu môi trường không có `python-dotenv`, notebook vẫn chạy được miễn là `GEOAPIFY_API_KEY` đã được export sẵn trong shell.


In [1]:
from pathlib import Path
import hashlib
import json
import math
import os
import time
import urllib.error
import urllib.parse
import urllib.request

import pandas as pd

try:
    from dotenv import load_dotenv
except ModuleNotFoundError:
    load_dotenv = None

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

if load_dotenv is not None:
    load_dotenv(ROOT / '.env', override=False)

DATA_DIR = ROOT / 'data'
GEOAPIFY_DIR = DATA_DIR / 'geoapify'
GEOAPIFY_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = GEOAPIFY_DIR / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = DATA_DIR / 'go_vap_tan_binh_100.json'
OUTPUT_PATH = GEOAPIFY_DIR / 'go_vap_tan_binh_100_enriched_geoapify_api.json'
CHECKPOINT_PATH = GEOAPIFY_DIR / 'go_vap_tan_binh_100_enriched_geoapify_api_checkpoint.json'
ERROR_LOG_PATH = GEOAPIFY_DIR / 'go_vap_tan_binh_100_geoapify_errors.json'

GEOAPIFY_API_URL = 'https://api.geoapify.com/v2/places'
GEOAPIFY_API_KEY = os.environ.get('GEOAPIFY_API_KEY')

if not GEOAPIFY_API_KEY:
    raise RuntimeError('Thiếu GEOAPIFY_API_KEY. Hãy export GEOAPIFY_API_KEY hoặc tạo file .env trước khi chạy notebook.')

INPUT_PATH, OUTPUT_PATH, CACHE_DIR

(PosixPath('/Users/totien/Documents/caohoc/Ki2-2026/IT2041_G8_DecisionMaking/data/go_vap_tan_binh_100.json'),
 PosixPath('/Users/totien/Documents/caohoc/Ki2-2026/IT2041_G8_DecisionMaking/data/geoapify/go_vap_tan_binh_100_enriched_geoapify_api.json'),
 PosixPath('/Users/totien/Documents/caohoc/Ki2-2026/IT2041_G8_DecisionMaking/data/geoapify/cache'))

## Bước 0.1. Cấu Hình Chạy

Mục đích:
- Thiết lập bán kính truy vấn, giới hạn kết quả, batch size và thời gian nghỉ giữa các batch.
- Chốt số lượng mục tiêu cần enrich.

Input:
- Các tham số cấu hình trong cell này.

Output:
- Bộ cấu hình dùng xuyên suốt notebook.


In [2]:
MAX_RADIUS_M = 1500
COUNT_RADIUS_M = 1000
LIMIT_PER_CATEGORY = 50
BASE_SLEEP_SECONDS = 0.3
RETRY_COUNT = 4
TIMEOUT_SECONDS = 60

START_INDEX = 0
TARGET_TOTAL_PROPERTIES = 100
MAX_PROPERTIES_PER_RUN = 15
BATCH_PAUSE_SECONDS = 10
RUN_UNTIL_DONE = True
MAX_BATCHES = 30
CONTINUE_ON_ERROR = True
FORCE_REFRESH = False
SAVE_CHECKPOINT_EVERY = 5

CATEGORY_ORDER = [
    'school',
    'park',
    'hospital',
    'supermarket',
    'market',
    'cafe',
    'boulevard',
]

CATEGORY_MAP = {
    'school': ['education.school', 'childcare.kindergarten'],
    'park': ['leisure.park'],
    'hospital': ['healthcare.hospital', 'healthcare.clinic_or_praxis.general'],
    'supermarket': ['commercial.supermarket'],
    'market': ['commercial.marketplace'],
    'cafe': ['catering.cafe'],
    'boulevard': ['highway.primary', 'highway.secondary', 'highway.trunk'],
}

CATEGORY_MAP

{'school': ['education.school', 'childcare.kindergarten'],
 'park': ['leisure.park'],
 'hospital': ['healthcare.hospital', 'healthcare.clinic_or_praxis.general'],
 'supermarket': ['commercial.supermarket'],
 'market': ['commercial.marketplace'],
 'cafe': ['catering.cafe'],
 'boulevard': ['highway.primary', 'highway.secondary', 'highway.trunk']}

## Bước 0.2. Nạp Dữ Liệu Gốc

Mục đích:
- Đọc bộ 100 bất động sản từ file JSON gốc.
- Xem nhanh dữ liệu đầu vào trước khi enrich.

Input:
- `data/go_vap_tan_binh_100.json`

Output:
- `properties`
- `DataFrame` preview đầu vào


In [3]:
with INPUT_PATH.open('r', encoding='utf-8') as f:
    properties = json.load(f)

df = pd.DataFrame(properties)
print('Số bản ghi:', len(properties))
display(df.head(3))
display(df[['district', 'price_billion_vnd', 'area_m2', 'bedrooms']].describe(include='all'))

Số bản ghi: 100


,property_id,title,district,ward,location_raw,price_million_vnd,price_billion_vnd,area_m2,price_per_m2_million,bedrooms,...,direction,position,latitude,longitude,description_snippet,distance_to_nearest_school_m,distance_to_nearest_park_m,distance_to_nearest_hospital_m,distance_to_nearest_supermarket_m,distance_to_nearest_boulevard_m
0,GV_001,"🔥 Giảm 440tr Nhà Quang Trung P10 GV, 20m2, 3.3x6m",Gò Vấp,Phường Gò Vấp,"Phường Gò Vấp,TP. Hồ Chí Minh(Mới)",1250.0,1.25,20.0,62.50,2,...,Nam,Trong hẻm,10.826903,106.673948,"🔥 Giảm 440tr Nhà Quang Trung P10 GV, 20m2, 3.3...",None,None,None,None,None
1,GV_002,⭐️ Giảm 50tr! Nhà Vuông SHR 2 tầng 26m2 – Quan...,Gò Vấp,Phường Thông Tây Hội,"Đường Quang Trung,Phường Thông Tây Hội,TP. Hồ ...",2850.0,2.85,26.0,109.62,2,...,NaN,Trong hẻm,10.842153,106.645882,⭐️ Giảm 50tr! Nhà Vuông SHR 2 tầng 26m2 – Quan...,None,None,None,None,None
2,GV_003,"Nhà 2 tầng sàn 52m2, Vuông – Lê Đức Thọ QV",Gò Vấp,Phường Gò Vấp,"Đường Lê Đức Thọ,Phường Gò Vấp,TP. Hồ Chí Mi...",3100.0,3.10,25.0,124.00,2,...,NaN,Đường chính,10.844656,106.674622,"Nhà 2 tầng sàn 52m2, Vuông – Lê Đức Thọ QV🏡 25...",None,None,None,None,None


,district,price_billion_vnd,area_m2,bedrooms
count,100,100.000000,100.00000,100.000000
unique,2,NaN,NaN,NaN
top,Gò Vấp,NaN,NaN,NaN
freq,50,NaN,NaN,NaN
mean,NaN,8.273000,69.10500,3.480000
std,NaN,4.103682,35.80164,1.445858
min,NaN,1.250000,20.00000,1.000000
25%,NaN,5.237500,44.45000,2.000000
50%,NaN,7.450000,62.50000,3.000000
75%,NaN,10.400000,83.50000,4.000000


## 1. Thiết Kế Truy Vấn Geoapify

Theo tài liệu chính thức của Geoapify Places API:
- endpoint là `https://api.geoapify.com/v2/places`
- dùng HTTP `GET`
- `categories` là tham số bắt buộc
- có thể dùng `filter=circle:lon,lat,radius`
- có thể thêm `bias=proximity:lon,lat` để ưu tiên kết quả gần điểm truy vấn

Ở notebook này, mỗi nhóm tiện ích sẽ gọi `1 request` riêng để dễ kiểm soát category và feature đầu ra.


## Bước 1. Xây Dựng Hàm Hỗ Trợ Và Hàm Gọi API

Mục đích:
- Tạo hàm tính khoảng cách Haversine.
- Tạo URL gọi Geoapify Places API.
- Hỗ trợ cache, retry, timeout.
- Chuẩn hóa kết quả Geoapify thành feature thống nhất với các pipeline khác.

Input:
- Tọa độ `latitude/longitude` của từng bất động sản.
- Category Geoapify tương ứng với từng nhóm tiện ích.

Output:
- Các hàm như `request_geoapify_places(...)`, `build_feature_block(...)`, `enrich_property_with_geoapify(...)`.


In [4]:
def haversine_m(lat1, lon1, lat2, lon2):
    earth_radius_m = 6371000
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return earth_radius_m * c


def build_cache_key(lat, lon, category_key, radius_m, limit):
    raw = f'{lat:.6f}|{lon:.6f}|{category_key}|{radius_m}|{limit}'
    return hashlib.md5(raw.encode('utf-8')).hexdigest()


def geoapify_request_url(lat, lon, category_key, radius_m=MAX_RADIUS_M, limit=LIMIT_PER_CATEGORY):
    categories = ','.join(CATEGORY_MAP[category_key])
    params = {
        'categories': categories,
        'filter': f'circle:{lon},{lat},{radius_m}',
        'bias': f'proximity:{lon},{lat}',
        'limit': limit,
        'apiKey': GEOAPIFY_API_KEY,
    }
    return f"{GEOAPIFY_API_URL}?{urllib.parse.urlencode(params)}"


def request_geoapify_places(lat, lon, category_key, radius_m=MAX_RADIUS_M, limit=LIMIT_PER_CATEGORY, force_refresh=False):
    cache_key = build_cache_key(lat, lon, category_key, radius_m, limit)
    cache_path = CACHE_DIR / f'{cache_key}.json'

    if cache_path.exists() and not force_refresh:
        with cache_path.open('r', encoding='utf-8') as f:
            return json.load(f)

    url = geoapify_request_url(lat, lon, category_key, radius_m=radius_m, limit=limit)
    req = urllib.request.Request(url, headers={'User-Agent': 'IT2041_G8_DecisionMaking/1.0 (geoapify notebook)'})

    last_error = None
    for attempt in range(1, RETRY_COUNT + 1):
        try:
            with urllib.request.urlopen(req, timeout=TIMEOUT_SECONDS) as resp:
                data = json.loads(resp.read().decode('utf-8'))
            with cache_path.open('w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            time.sleep(BASE_SLEEP_SECONDS)
            return data
        except urllib.error.HTTPError as exc:
            last_error = exc
            wait_s = attempt * 5
            print(f'HTTP lỗi cho {category_key} tại ({lat}, {lon}): {exc}. Thử lại sau {wait_s}s')
            time.sleep(wait_s)
        except (urllib.error.URLError, TimeoutError, json.JSONDecodeError) as exc:
            last_error = exc
            wait_s = attempt * 5
            print(f'Lỗi gọi Geoapify cho {category_key} tại ({lat}, {lon}): {exc}. Thử lại sau {wait_s}s')
            time.sleep(wait_s)

    raise RuntimeError(f'Geoapify request failed for {category_key} after {RETRY_COUNT} attempts: {last_error}')


def normalize_geoapify_features(raw, property_lat, property_lon):
    items = []
    for feature in raw.get('features', []):
        props = feature.get('properties', {})
        lon = props.get('lon')
        lat = props.get('lat')
        if lat is None or lon is None:
            continue
        distance_m = props.get('distance')
        if distance_m is None:
            distance_m = round(haversine_m(property_lat, property_lon, lat, lon))
        items.append({
            'name': props.get('name') or props.get('formatted') or f"geoapify_{props.get('place_id', 'unknown')}",
            'lat': lat,
            'lon': lon,
            'distance_m': round(distance_m),
            'place_id': props.get('place_id'),
            'categories': props.get('categories', []),
            'raw_properties': props,
        })
    items.sort(key=lambda x: x['distance_m'])
    return items


def build_feature_block(grouped, count_radius_m=COUNT_RADIUS_M):
    features = {}
    for category in CATEGORY_ORDER:
        items = grouped.get(category, [])
        if items:
            features[f'distance_to_nearest_{category}_m'] = items[0]['distance_m']
            features[f'nearest_{category}_name'] = items[0]['name']
            features[f'near_{category}_count_1km'] = sum(item['distance_m'] <= count_radius_m for item in items)
        else:
            features[f'distance_to_nearest_{category}_m'] = None
            features[f'nearest_{category}_name'] = None
            features[f'near_{category}_count_1km'] = 0
    return features


def enrich_property_with_geoapify(prop, force_refresh=False):
    lat = prop['latitude']
    lon = prop['longitude']
    grouped = {}
    total_api_results = 0

    for category in CATEGORY_ORDER:
        raw = request_geoapify_places(lat, lon, category, force_refresh=force_refresh)
        items = normalize_geoapify_features(raw, lat, lon)
        grouped[category] = items
        total_api_results += len(raw.get('features', []))

    features = build_feature_block(grouped, count_radius_m=COUNT_RADIUS_M)
    enriched = dict(prop)
    enriched.update(features)
    enriched['enrichment_provider'] = 'geoapify_places_api'
    enriched['enrichment_radius_m'] = MAX_RADIUS_M
    enriched['enrichment_count_radius_m'] = COUNT_RADIUS_M
    enriched['api_result_count'] = total_api_results
    return enriched

## Bước 1.1. Thử Một Lần Gọi API Mẫu

Mục đích:
- Test nhanh một vài category trên một căn mẫu.
- Kiểm tra API key, endpoint và schema response.

Input:
- `properties[0]`

Output:
- response mẫu cho một nhóm category
- số lượng kết quả và một vài field đầu tiên


In [5]:
sample_prop = properties[0]
sample_raw = request_geoapify_places(sample_prop['latitude'], sample_prop['longitude'], 'school')
print(sample_prop['property_id'], sample_prop['district'])
print('Số features từ API:', len(sample_raw.get('features', [])))
sample_raw.get('features', [])[:2]

GV_001 Gò Vấp
Số features từ API: 16


[{'type': 'Feature',
  'properties': {'name': 'Trường Trung học cơ sở&Trung học phổ thông Hồng Hà',
   'country': 'Vietnam',
   'country_code': 'vn',
   'city': 'Thuận An',
   'postcode': '70048',
   'suburb': 'Phường Gò Vấp',
   'street': 'Hẻm 251 Quang Trung',
   'housenumber': '570',
   'iso3166_2': 'VN-SG',
   'lon': 106.6731667,
   'lat': 10.8287154,
   'formatted': 'Trường Trung học cơ sở&Trung học phổ thông Hồng Hà, 570, Hẻm 251 Quang Trung, Phường Gò Vấp, Thuận An, 70048, Vietnam',
   'address_line1': 'Trường Trung học cơ sở&Trung học phổ thông Hồng Hà',
   'address_line2': '570, Hẻm 251 Quang Trung, Phường Gò Vấp, Thuận An, 70048, Vietnam',
   'categories': ['education', 'education.school'],
   'details': [],
   'datasource': {'sourcename': 'openstreetmap',
    'attribution': '© OpenStreetMap contributors',
    'license': 'Open Database License',
    'url': 'https://www.openstreetmap.org/copyright',
    'raw': {'lat': 10.8287154,
     'lon': 106.6731667,
     'name': 'Trường T

## 2. Chạy Toàn Bộ Quá Trình Enrich

Notebook này chạy tương tự bản Overpass nhưng dùng Geoapify Places API.

Cách chạy:
- chạy theo batch `15` căn
- ghi checkpoint/output giữa chừng
- resume từ checkpoint nếu đã có dữ liệu cũ
- tiếp tục loop cho tới khi đủ `100` căn mục tiêu

Input:
- `properties`
- API key Geoapify
- cấu hình batch/target

Output:
- file output JSON, checkpoint và error log trong `data/geoapify/`


In [6]:
if CHECKPOINT_PATH.exists() and not FORCE_REFRESH:
    with CHECKPOINT_PATH.open('r', encoding='utf-8') as f:
        enriched_properties = json.load(f)
    print(f'Đã nạp checkpoint có sẵn với {len(enriched_properties)} bản ghi')
else:
    enriched_properties = []

if ERROR_LOG_PATH.exists() and not FORCE_REFRESH:
    with ERROR_LOG_PATH.open('r', encoding='utf-8') as f:
        error_logs = json.load(f)
else:
    error_logs = []

all_target_properties = properties[START_INDEX: START_INDEX + TARGET_TOTAL_PROPERTIES]
target_property_ids = {prop['property_id'] for prop in all_target_properties}
existing_ids = {item['property_id'] for item in enriched_properties if item['property_id'] in target_property_ids}
batch_counter = 0

print(f'Mục tiêu enrich: {len(all_target_properties)} bất động sản')
print(f'Đã có sẵn trong checkpoint: {len(existing_ids)} bất động sản')

while True:
    remaining_properties = [
        prop for prop in all_target_properties
        if FORCE_REFRESH or prop['property_id'] not in existing_ids
    ]

    if not remaining_properties or len(existing_ids) >= len(target_property_ids):
        print('Đã hoàn tất toàn bộ phạm vi mục tiêu.')
        break

    if not RUN_UNTIL_DONE and batch_counter >= 1:
        print('RUN_UNTIL_DONE = False nên dừng sau 1 batch.')
        break

    if batch_counter >= MAX_BATCHES:
        print(f'Đã chạm giới hạn MAX_BATCHES = {MAX_BATCHES}. Dừng để tránh chạy quá lâu.')
        break

    current_batch = remaining_properties[:MAX_PROPERTIES_PER_RUN]
    batch_counter += 1
    processed_in_this_batch = 0

    print(f'\n=== Batch {batch_counter} ===')
    print(f'Số bản ghi còn lại trong mục tiêu: {len(remaining_properties)}')
    print(f'Batch này sẽ xử lý {len(current_batch)} căn')
    print('Danh sách property_id:', [prop['property_id'] for prop in current_batch])

    for prop in current_batch:
        try:
            enriched = enrich_property_with_geoapify(prop, force_refresh=FORCE_REFRESH)
            enriched_properties = [item for item in enriched_properties if item['property_id'] != prop['property_id']]
            enriched_properties.append(enriched)
            existing_ids.add(prop['property_id'])
            processed_in_this_batch += 1
            print(f"Xong {prop['property_id']} | API elements: {enriched['api_result_count']}")

            if processed_in_this_batch % SAVE_CHECKPOINT_EVERY == 0:
                snapshot = sorted(enriched_properties, key=lambda x: x['property_id'])
                with CHECKPOINT_PATH.open('w', encoding='utf-8') as f:
                    json.dump(snapshot, f, ensure_ascii=False, indent=2)
                with OUTPUT_PATH.open('w', encoding='utf-8') as f:
                    json.dump(snapshot, f, ensure_ascii=False, indent=2)
                print(f'Đã lưu checkpoint/output với {len(existing_ids)} / {len(target_property_ids)} bản ghi mục tiêu')

        except Exception as exc:
            error_logs.append({
                'property_id': prop['property_id'],
                'district': prop['district'],
                'latitude': prop['latitude'],
                'longitude': prop['longitude'],
                'error': str(exc),
                'batch': batch_counter,
            })
            print(f"Lỗi tại {prop['property_id']}: {exc}")
            if not CONTINUE_ON_ERROR:
                raise

    enriched_properties = sorted(enriched_properties, key=lambda x: x['property_id'])
    with CHECKPOINT_PATH.open('w', encoding='utf-8') as f:
        json.dump(enriched_properties, f, ensure_ascii=False, indent=2)
    with OUTPUT_PATH.open('w', encoding='utf-8') as f:
        json.dump(enriched_properties, f, ensure_ascii=False, indent=2)
    with ERROR_LOG_PATH.open('w', encoding='utf-8') as f:
        json.dump(error_logs, f, ensure_ascii=False, indent=2)

    print(f'Hoàn tất batch {batch_counter}')
    print('Số bản ghi enrich trong mục tiêu hiện có:', len(existing_ids))
    print('Tổng số lỗi tích lũy:', len(error_logs))

    remaining_after_batch = [
        prop for prop in all_target_properties
        if FORCE_REFRESH or prop['property_id'] not in existing_ids
    ]
    if not remaining_after_batch:
        print('Đã enrich xong toàn bộ các căn trong phạm vi mục tiêu.')
        break

    if RUN_UNTIL_DONE:
        print(f'Nghỉ {BATCH_PAUSE_SECONDS}s trước khi chạy batch tiếp theo...')
        time.sleep(BATCH_PAUSE_SECONDS)
    else:
        break

enriched_properties = sorted(
    [item for item in enriched_properties if item['property_id'] in target_property_ids],
    key=lambda x: x['property_id']
)
with OUTPUT_PATH.open('w', encoding='utf-8') as f:
    json.dump(enriched_properties, f, ensure_ascii=False, indent=2)
with ERROR_LOG_PATH.open('w', encoding='utf-8') as f:
    json.dump(error_logs, f, ensure_ascii=False, indent=2)

print('\nHoàn tất toàn bộ lượt chạy notebook')
print('Tổng số bản ghi enrich trong mục tiêu:', len(enriched_properties))
print(f'File output hiện tại: {OUTPUT_PATH}')
print('Tổng số lỗi:', len(error_logs))

Đã nạp checkpoint có sẵn với 25 bản ghi
Mục tiêu enrich: 100 bất động sản
Đã có sẵn trong checkpoint: 25 bất động sản

=== Batch 1 ===
Số bản ghi còn lại trong mục tiêu: 75
Batch này sẽ xử lý 15 căn
Danh sách property_id: ['GV_026', 'GV_027', 'GV_028', 'GV_029', 'GV_030', 'GV_031', 'GV_032', 'GV_033', 'GV_034', 'GV_035', 'GV_036', 'GV_037', 'GV_038', 'GV_039', 'GV_040']
Xong GV_026 | API elements: 117
Xong GV_027 | API elements: 135


Xong GV_028 | API elements: 84


Xong GV_029 | API elements: 95


Xong GV_030 | API elements: 102
Đã lưu checkpoint/output với 30 / 100 bản ghi mục tiêu


Xong GV_031 | API elements: 112


...

Xong TB_046 | API elements: 91


Xong TB_047 | API elements: 200


Xong TB_048 | API elements: 74


Xong TB_049 | API elements: 155


Xong TB_050 | API elements: 169
Đã lưu checkpoint/output với 100 / 100 bản ghi mục tiêu
Hoàn tất batch 5
Số bản ghi enrich trong mục tiêu hiện có: 100
Tổng số lỗi tích lũy: 0
Đã enrich xong toàn bộ các căn trong phạm vi mục tiêu.

Hoàn tất toàn bộ lượt chạy notebook
Tổng số bản ghi enrich trong mục tiêu: 100
File output hiện tại: /Users/totien/Documents/caohoc/Ki2-2026/IT2041_G8_DecisionMaking/data/geoapify/go_vap_tan_binh_100_enriched_geoapify_api.json
Tổng số lỗi: 0


## 3. Kiểm Tra Chất Lượng

Mục đích:
- Xem nhanh phân bố các field enrich.
- Kiểm tra độ đầy của dữ liệu sau khi chạy.

Input:
- `enriched_properties`

Output:
- `enriched_df` và các bảng thống kê mô tả.


In [7]:
enriched_df = pd.DataFrame(enriched_properties)
distance_cols = [col for col in enriched_df.columns if col.startswith('distance_to_nearest_')]
count_cols = [col for col in enriched_df.columns if col.startswith('near_') and col.endswith('_count_1km')]

print('Số record hiện có:', len(enriched_df))
if len(enriched_df) > 0:
    display(enriched_df[distance_cols].describe().T)
    display(enriched_df[count_cols].describe().T)
    display(enriched_df.groupby('district')[distance_cols].mean().round(1))

Số record hiện có: 100


,count,mean,std,min,25%,50%,75%,max
distance_to_nearest_school_m,100.0,258.080000,143.846206,40.0,139.75,234.5,352.00,681.0
distance_to_nearest_park_m,100.0,469.350000,260.428449,57.0,289.00,410.0,600.75,1176.0
distance_to_nearest_hospital_m,73.0,729.178082,371.574030,99.0,456.00,680.0,1038.00,1471.0
distance_to_nearest_supermarket_m,98.0,724.755102,302.185103,59.0,520.25,745.0,933.25,1339.0
distance_to_nearest_boulevard_m,100.0,221.830000,214.014446,1.0,80.00,162.0,280.75,972.0
distance_to_nearest_market_m,97.0,578.525773,353.534330,105.0,303.00,503.0,744.00,1498.0
distance_to_nearest_cafe_m,100.0,354.980000,218.573381,21.0,206.75,308.5,468.00,1107.0


,count,mean,std,min,25%,50%,75%,max
near_school_count_1km,100.0,13.89,8.518423,2.0,6.0,12.0,21.00,30.0
near_park_count_1km,100.0,6.39,4.287390,0.0,2.0,6.0,10.00,17.0
near_hospital_count_1km,100.0,1.02,1.310255,0.0,0.0,1.0,2.00,6.0
near_supermarket_count_1km,100.0,1.37,1.440083,0.0,1.0,1.0,2.00,7.0
near_market_count_1km,100.0,2.03,1.684481,0.0,1.0,1.0,3.25,6.0
near_cafe_count_1km,100.0,10.45,9.031231,0.0,4.0,10.0,13.00,50.0
near_boulevard_count_1km,100.0,36.12,14.756803,1.0,26.0,38.5,50.00,50.0


,distance_to_nearest_school_m,distance_to_nearest_park_m,distance_to_nearest_hospital_m,distance_to_nearest_supermarket_m,distance_to_nearest_boulevard_m,distance_to_nearest_market_m,distance_to_nearest_cafe_m
district,,,,,,,
Gò Vấp,286.9,568.9,895.9,722.5,210.6,753.1,331.4
Tân Bình,229.3,369.8,599.0,726.9,233.1,414.5,378.6


## 4. Lưu Kết Quả

Mục đích:
- Xác nhận lại file output cuối đã được ghi vào `data/geoapify/`.

Input:
- `enriched_properties`

Output:
- File `data/geoapify/go_vap_tan_binh_100_enriched_geoapify_api.json`
- File checkpoint và error log tương ứng


In [8]:
print('Output file:', OUTPUT_PATH)
print('Checkpoint file:', CHECKPOINT_PATH)
print('Error log file:', ERROR_LOG_PATH)

Output file: /Users/totien/Documents/caohoc/Ki2-2026/IT2041_G8_DecisionMaking/data/geoapify/go_vap_tan_binh_100_enriched_geoapify_api.json
Checkpoint file: /Users/totien/Documents/caohoc/Ki2-2026/IT2041_G8_DecisionMaking/data/geoapify/go_vap_tan_binh_100_enriched_geoapify_api_checkpoint.json
Error log file: /Users/totien/Documents/caohoc/Ki2-2026/IT2041_G8_DecisionMaking/data/geoapify/go_vap_tan_binh_100_geoapify_errors.json


## 5. Gợi Ý Tích Hợp Vào Project

- Notebook này phù hợp khi nhóm muốn thử một provider thương mại ổn định hơn Overpass.
- Cần quản lý API key cẩn thận qua biến môi trường.
- Có thể so sánh trực tiếp output của `Geoapify` với `Overpass` vì schema feature được giữ gần giống nhau.
- Nếu muốn benchmark công bằng, hãy chạy cùng một tập `100` căn và so sánh độ đầy dữ liệu, độ ổn định và chi phí gọi API.
